# ING Hubs Datathon 2025 - Churn Prediction Vol.3
## Advanced Feature Engineering with Polars

Bu notebook, BTK Datathon 2025 kazanan çözümünden esinlenerek:
- **Polars** kullanarak hızlı veri işleme
- **Leave-one-out** encoding ile data leakage önleme
- **Historical features** ile time-based özellikler
- **Model blending** ile ensemble

---

## 1. Kütüphaneleri İçe Aktarma

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.ensemble import VotingClassifier

# Models
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("crest")
pd.set_option('display.max_columns', None)

print("✅ Tüm kütüphaneler yüklendi!")
print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Tüm kütüphaneler yüklendi!
📅 2025-10-12 15:06:17


## 2. Veri Yükleme (Polars ile)

In [2]:
print("="*80)
print("VERİLERİ YÜKLEME (POLARS)")
print("="*80)

# Polars ile hızlı yükleme
customer_history_pl = pl.read_csv('data/customer_history.csv').sort(["cust_id", "date"])
customers_pl = pl.read_csv('data/customers.csv')
reference_data_pl = pl.read_csv('data/referance_data.csv')
reference_data_test_pl = pl.read_csv('data/referance_data_test.csv')

# Tarihleri datetime'a çevir
customer_history_pl = customer_history_pl.with_columns(
    pl.col("date").str.to_datetime("%Y-%m-%d")
)
reference_data_pl = reference_data_pl.with_columns(
    pl.col("ref_date").str.to_datetime("%Y-%m-%d")
)
reference_data_test_pl = reference_data_test_pl.with_columns(
    pl.col("ref_date").str.to_datetime("%Y-%m-%d")
)

print(f"\n✅ Veriler yüklendi (Polars):")
print(f"   📊 Customer History: {customer_history_pl.shape}")
print(f"   📊 Customers: {customers_pl.shape}")
print(f"   📊 Train: {reference_data_pl.shape}")
print(f"   📊 Test: {reference_data_test_pl.shape}")

print(f"\n🎯 Churn dağılımı:")
print(reference_data_pl['churn'].value_counts().sort('churn'))

VERİLERİ YÜKLEME (POLARS)

✅ Veriler yüklendi (Polars):
   📊 Customer History: (5359609, 7)
   📊 Customers: (176293, 8)
   📊 Train: (133287, 3)
   📊 Test: (43006, 2)

🎯 Churn dağılımı:
shape: (2, 2)
┌───────┬────────┐
│ churn ┆ count  │
│ ---   ┆ ---    │
│ i64   ┆ u32    │
╞═══════╪════════╡
│ 0     ┆ 114417 │
│ 1     ┆ 18870  │
└───────┴────────┘

✅ Veriler yüklendi (Polars):
   📊 Customer History: (5359609, 7)
   📊 Customers: (176293, 8)
   📊 Train: (133287, 3)
   📊 Test: (43006, 2)

🎯 Churn dağılımı:
shape: (2, 2)
┌───────┬────────┐
│ churn ┆ count  │
│ ---   ┆ ---    │
│ i64   ┆ u32    │
╞═══════╪════════╡
│ 0     ┆ 114417 │
│ 1     ┆ 18870  │
└───────┴────────┘


## 3. Train ve Test Birleştirme

In [3]:
# Train ve test'i birleştir (split column ile ayır)
train_with_split = reference_data_pl.with_columns(pl.lit("train").alias("split"))
test_with_split = reference_data_test_pl.with_columns(pl.lit("test").alias("split"))

# Full reference (churn olmadan test için)
full_reference = pl.concat([
    train_with_split.drop("churn"),
    test_with_split
])

# Test session listesi
test_customers = test_with_split["cust_id"].unique().to_list()
train_customers = train_with_split["cust_id"].unique().to_list()

print(f"✅ Train-Test birleştirildi")
print(f"   Train customers: {len(train_customers):,}")
print(f"   Test customers: {len(test_customers):,}")

✅ Train-Test birleştirildi
   Train customers: 133,287
   Test customers: 43,006


## 4. Leave-One-Out Encoding for Categories

**Target Encoding** ile data leakage önleme:
- Her müşteri için kendi churn değeri **hariç** diğerlerinin ortalaması
- Province bazında churn oranları
- Work sector bazında churn oranları

In [4]:
def calculate_loo_encoding(train_df, category_col):
    """
    Leave-one-out encoding for categorical variables
    Her müşteri için kendi değeri hariç kategori ortalaması
    """
    print(f"🔧 LOO encoding: {category_col}")
    
    # Global kategori istatistikleri
    category_stats = train_df.group_by(category_col).agg([
        pl.col("churn").sum().alias(f"{category_col}_total_churn"),
        pl.col("churn").count().alias(f"{category_col}_total_count")
    ])
    
    # Join ve LOO hesaplama
    result = (
        train_df
        .join(category_stats, on=category_col, how="left")
        .with_columns([
            # Leave-one-out mean
            ((pl.col(f"{category_col}_total_churn") - pl.col("churn")) / 
             pl.max_horizontal(pl.col(f"{category_col}_total_count") - 1, 1)).alias(f"{category_col}_churn_rate_loo"),
            
            # Count (excluding current)
            (pl.col(f"{category_col}_total_count") - 1).alias(f"{category_col}_count_loo")
        ])
        .select(["cust_id", f"{category_col}_churn_rate_loo", f"{category_col}_count_loo"])
    )
    
    print(f"  ✅ {category_col} LOO encoding tamamlandı")
    return result

# Province ve work_sector için LOO encoding
province_loo = calculate_loo_encoding(
    train_with_split.join(customers_pl, on="cust_id", how="left"),
    "province"
)

work_sector_loo = calculate_loo_encoding(
    train_with_split.join(customers_pl, on="cust_id", how="left"),
    "work_sector"
)

print("\n✅ LOO encodings hazır!")

🔧 LOO encoding: province
  ✅ province LOO encoding tamamlandı
🔧 LOO encoding: work_sector
  ✅ work_sector LOO encoding tamamlandı

✅ LOO encodings hazır!


## 5. Feature Engineering Fonksiyonları

### 5.1 Time-Based Features (Customer History)

In [5]:
def create_customer_features(history_df, ref_df):
    """
    Customer history'den time-based features
    Referans tarihinden ÖNCEKİ veriler kullanılır
    """
    print("🔧 Customer features oluşturuluyor...")
    
    # Referans tarihlerini ekle
    df = history_df.join(ref_df.select(["cust_id", "ref_date"]), on="cust_id", how="inner")
    
    # Sadece geçmiş verileri al
    df = df.filter(pl.col("date") < pl.col("ref_date"))
    
    print(f"  ✓ Geçmiş kayıtlar: {len(df):,}")
    
    # Days before reference
    df = df.with_columns(
        (pl.col("ref_date") - pl.col("date")).dt.total_days().alias("days_before_ref")
    )
    
    # Time windows
    df = df.with_columns([
        (pl.col("days_before_ref") <= 30).alias("is_last_30d"),
        (pl.col("days_before_ref") <= 90).alias("is_last_90d"),
        (pl.col("days_before_ref") <= 180).alias("is_last_180d"),
    ])
    
    # Aggregate features per window
    features_list = []
    
    for window, flag in [("30d", "is_last_30d"), ("90d", "is_last_90d"), ("180d", "is_last_180d")]:
        print(f"  ⏳ {window} penceresi...")
        
        window_features = df.filter(pl.col(flag) == True).group_by("cust_id").agg([
            # EFT features
            pl.col("mobile_eft_all_cnt").sum().alias(f"eft_cnt_sum_{window}"),
            pl.col("mobile_eft_all_cnt").mean().alias(f"eft_cnt_mean_{window}"),
            pl.col("mobile_eft_all_cnt").max().alias(f"eft_cnt_max_{window}"),
            pl.col("mobile_eft_all_cnt").std().alias(f"eft_cnt_std_{window}"),
            
            pl.col("mobile_eft_all_amt").sum().alias(f"eft_amt_sum_{window}"),
            pl.col("mobile_eft_all_amt").mean().alias(f"eft_amt_mean_{window}"),
            
            # CC features
            pl.col("cc_transaction_all_cnt").sum().alias(f"cc_cnt_sum_{window}"),
            pl.col("cc_transaction_all_cnt").mean().alias(f"cc_cnt_mean_{window}"),
            pl.col("cc_transaction_all_amt").sum().alias(f"cc_amt_sum_{window}"),
            pl.col("cc_transaction_all_amt").mean().alias(f"cc_amt_mean_{window}"),
            
            # Active products
            pl.col("active_product_category_nbr").mean().alias(f"active_prod_mean_{window}"),
            pl.col("active_product_category_nbr").max().alias(f"active_prod_max_{window}"),
            
            # Activity count
            pl.col("date").count().alias(f"months_active_{window}"),
        ])
        
        features_list.append(window_features)
    
    # Overall features
    print("  📊 Genel özellikler...")
    overall_features = df.group_by("cust_id").agg([
        pl.col("date").count().alias("total_months_history"),
        pl.col("date").max().alias("last_transaction_date"),
        pl.col("mobile_eft_all_cnt").sum().alias("eft_cnt_total"),
        pl.col("cc_transaction_all_cnt").sum().alias("cc_cnt_total"),
        pl.col("mobile_eft_all_amt").sum().alias("eft_amt_total"),
        pl.col("cc_transaction_all_amt").sum().alias("cc_amt_total"),
    ])
    
    # Recency
    recency_df = df.group_by("cust_id").agg(
        pl.col("date").max().alias("last_transaction_date")
    ).join(ref_df.select(["cust_id", "ref_date"]), on="cust_id").with_columns(
        (pl.col("ref_date") - pl.col("last_transaction_date")).dt.total_days().alias("recency_days")
    ).select(["cust_id", "recency_days"])
    
    overall_features = overall_features.join(recency_df, on="cust_id", how="left").drop("last_transaction_date")
    
    # Merge all
    result = ref_df.select(["cust_id"])
    for feat in features_list:
        result = result.join(feat, on="cust_id", how="left")
    result = result.join(overall_features, on="cust_id", how="left")
    
    # Fill nulls
    result = result.fill_null(0)
    
    print(f"  ✅ {len(result):,} müşteri için {len(result.columns)-1} özellik")
    
    return result

print("✅ Fonksiyon tanımlandı!")

✅ Fonksiyon tanımlandı!


### 5.2 Derived Features

In [6]:
def create_derived_features(df):
    """
    Hesaplanmış özellikler (oranlar, skorlar vb.)
    """
    print("🔧 Derived features oluşturuluyor...")
    
    # Step 1: Basic derived features
    df = df.with_columns([
        # Total activity
        (pl.col("eft_cnt_sum_90d") + pl.col("cc_cnt_sum_90d")).alias("total_transactions_90d"),
        (pl.col("eft_amt_sum_90d") + pl.col("cc_amt_sum_90d")).alias("total_amount_90d"),
        
        # Ratios
        (pl.col("eft_cnt_sum_90d") / (pl.col("cc_cnt_sum_90d") + 1)).alias("eft_cc_ratio_90d"),
        (pl.col("eft_amt_sum_90d") / (pl.col("cc_amt_sum_90d") + 1)).alias("eft_cc_amt_ratio_90d"),
        
        # Average transaction value
        (pl.col("eft_amt_sum_90d") / (pl.col("eft_cnt_sum_90d") + 1)).alias("avg_eft_value_90d"),
        (pl.col("cc_amt_sum_90d") / (pl.col("cc_cnt_sum_90d") + 1)).alias("avg_cc_value_90d"),
        
        # Recency risk
        (pl.col("recency_days") / 30).alias("recency_risk"),
        
        # Inactivity
        (pl.col("months_active_90d") == 0).cast(pl.Int32).alias("is_inactive_90d"),
        
        # Activity score (weighted)
        (pl.col("eft_cnt_sum_90d") * 2 + pl.col("cc_cnt_sum_90d") * 3).alias("activity_score_90d"),
    ])
    
    # Step 2: Features that depend on previous derived columns
    df = df.with_columns([
        # Activity decline (uses total_transactions_90d from step 1)
        (pl.col("total_transactions_90d") / (pl.col("eft_cnt_sum_180d") + pl.col("cc_cnt_sum_180d") + 1)).alias("activity_recent_ratio"),
    ])
    
    print(f"  ✅ Derived features eklendi")
    return df

print("✅ Fonksiyon tanımlandı!")

✅ Fonksiyon tanımlandı!


## 6. Feature Engineering Pipeline

In [7]:
print("="*80)
print("TRAIN SET - FEATURE ENGINEERING")
print("="*80)

# Customer history features
train_cust_features = create_customer_features(customer_history_pl, train_with_split)

# Derived features
train_cust_features = create_derived_features(train_cust_features)

# Demographic features
print("\n🔧 Demografik özellikler ekleniyor...")
train_features = (
    train_with_split
    .join(train_cust_features, on="cust_id", how="left")
    .join(customers_pl, on="cust_id", how="left")
    .join(province_loo, on="cust_id", how="left")
    .join(work_sector_loo, on="cust_id", how="left")
)

print(f"\n✅ Train features hazır: {train_features.shape}")
train_features.head(3)

TRAIN SET - FEATURE ENGINEERING
🔧 Customer features oluşturuluyor...
  ✓ Geçmiş kayıtlar: 3,527,588
  ⏳ 30d penceresi...
  ✓ Geçmiş kayıtlar: 3,527,588
  ⏳ 30d penceresi...
  ⏳ 90d penceresi...
  ⏳ 180d penceresi...
  ⏳ 90d penceresi...
  ⏳ 180d penceresi...
  📊 Genel özellikler...
  ✅ 133,287 müşteri için 45 özellik
🔧 Derived features oluşturuluyor...
  ✅ Derived features eklendi

🔧 Demografik özellikler ekleniyor...

✅ Train features hazır: (133287, 70)
  📊 Genel özellikler...
  ✅ 133,287 müşteri için 45 özellik
🔧 Derived features oluşturuluyor...
  ✅ Derived features eklendi

🔧 Demografik özellikler ekleniyor...

✅ Train features hazır: (133287, 70)


cust_id,ref_date,churn,split,eft_cnt_sum_30d,eft_cnt_mean_30d,eft_cnt_max_30d,eft_cnt_std_30d,eft_amt_sum_30d,eft_amt_mean_30d,cc_cnt_sum_30d,cc_cnt_mean_30d,cc_amt_sum_30d,cc_amt_mean_30d,active_prod_mean_30d,active_prod_max_30d,months_active_30d,eft_cnt_sum_90d,eft_cnt_mean_90d,eft_cnt_max_90d,eft_cnt_std_90d,eft_amt_sum_90d,eft_amt_mean_90d,cc_cnt_sum_90d,cc_cnt_mean_90d,cc_amt_sum_90d,cc_amt_mean_90d,active_prod_mean_90d,active_prod_max_90d,months_active_90d,eft_cnt_sum_180d,eft_cnt_mean_180d,eft_cnt_max_180d,eft_cnt_std_180d,eft_amt_sum_180d,eft_amt_mean_180d,cc_cnt_sum_180d,cc_cnt_mean_180d,cc_amt_sum_180d,cc_amt_mean_180d,active_prod_mean_180d,active_prod_max_180d,months_active_180d,total_months_history,eft_cnt_total,cc_cnt_total,eft_amt_total,cc_amt_total,recency_days,total_transactions_90d,total_amount_90d,eft_cc_ratio_90d,eft_cc_amt_ratio_90d,avg_eft_value_90d,avg_cc_value_90d,recency_risk,is_inactive_90d,activity_score_90d,activity_recent_ratio,gender,age,province,religion,work_type,work_sector,tenure,province_churn_rate_loo,province_count_loo,work_sector_churn_rate_loo,work_sector_count_loo
i64,datetime[μs],i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,u32,u32,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,str,i64,str,str,str,str,i64,f64,u32,f64,u32
0,2017-09-01 00:00:00,0,"""train""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,4.0,2.0,2.0,0.0,478.34,239.17,0.0,0.0,0.0,0.0,2.0,2,2,11.0,2.2,3.0,0.447214,830.86,166.172,0.0,0.0,0.0,0.0,2.0,2,5,20,46.0,0.0,2365.69,0.0,31,4.0,478.34,4.0,478.34,95.668,0.0,1.033333,0,8.0,0.333333,"""F""",64,"""NOH""","""U""","""Part-time""","""Technology""",135,0.14278,22181,0.141247,17565
3,2018-10-01 00:00:00,0,"""train""",1.0,1.0,1.0,0.0,333.07,333.07,17.0,17.0,779.4,779.4,3.0,3,1,2.0,1.0,1.0,0.0,567.47,283.735,37.0,18.5,1962.86,981.43,3.0,3,2,5.0,1.0,1.0,0.0,1051.84,210.368,99.0,19.8,3257.6,651.52,3.0,3,5,33,55.0,814.0,1971.62,20077.48,30,39.0,2530.33,0.052632,0.288956,189.156667,51.654211,1.0,0,115.0,0.371429,"""F""",22,"""ZUI""","""C""","""Student""",null,47,0.14047,28490,null,null
5,2018-03-01 00:00:00,1,"""train""",5.0,5.0,5.0,0.0,1390.78,1390.78,2.0,2.0,10.86,10.86,2.0,2,1,12.0,4.0,5.0,1.0,2979.79,993.263333,9.0,3.0,45.3,15.1,2.0,2,3,19.0,3.8,5.0,0.83666,4174.02,834.804,13.0,2.6,76.11,15.222,2.0,2,5,26,65.0,350.0,9710.94,2467.23,28,21.0,3025.09,1.2,64.358315,229.214615,4.53,0.933333,0,51.0,0.636364,"""M""",27,"""ZUI""","""U""","""Full-time""","""Finance""",108,0.140435,28490,0.140103,12655


In [10]:
print("="*80)
print("TEST SET - FEATURE ENGINEERING")
print("="*80)

# Customer history features
test_cust_features = create_customer_features(customer_history_pl, test_with_split)

# Derived features
test_cust_features = create_derived_features(test_cust_features)

# For test, use global category means (no LOO)
province_global = (
    train_with_split.join(customers_pl, on="cust_id", how="left")
    .group_by("province").agg([
        pl.col("churn").mean().alias("province_churn_rate_loo"),
        pl.col("churn").count().alias("province_count_loo")
    ])
)

work_sector_global = (
    train_with_split.join(customers_pl, on="cust_id", how="left")
    .group_by("work_sector").agg([
        pl.col("churn").mean().alias("work_sector_churn_rate_loo"),
        pl.col("churn").count().alias("work_sector_count_loo")
    ])
)

# Demographic features
print("\n🔧 Demografik özellikler ekleniyor...")
test_features = (
    test_with_split
    .join(test_cust_features, on="cust_id", how="left")
    .join(customers_pl, on="cust_id", how="left")
    .join(province_global, on="province", how="left")
    .join(work_sector_global, on="work_sector", how="left")
)

print(f"\n✅ Test features hazır: {test_features.shape}")
test_features.head(3)

TEST SET - FEATURE ENGINEERING
🔧 Customer features oluşturuluyor...
  ✓ Geçmiş kayıtlar: 1,655,728
  ⏳ 30d penceresi...
  ⏳ 90d penceresi...
  ⏳ 180d penceresi...
  📊 Genel özellikler...
  ✅ 43,006 müşteri için 45 özellik
🔧 Derived features oluşturuluyor...
  ✅ Derived features eklendi

🔧 Demografik özellikler ekleniyor...

✅ Test features hazır: (43006, 69)
  ✅ 43,006 müşteri için 45 özellik
🔧 Derived features oluşturuluyor...
  ✅ Derived features eklendi

🔧 Demografik özellikler ekleniyor...

✅ Test features hazır: (43006, 69)


cust_id,ref_date,split,eft_cnt_sum_30d,eft_cnt_mean_30d,eft_cnt_max_30d,eft_cnt_std_30d,eft_amt_sum_30d,eft_amt_mean_30d,cc_cnt_sum_30d,cc_cnt_mean_30d,cc_amt_sum_30d,cc_amt_mean_30d,active_prod_mean_30d,active_prod_max_30d,months_active_30d,eft_cnt_sum_90d,eft_cnt_mean_90d,eft_cnt_max_90d,eft_cnt_std_90d,eft_amt_sum_90d,eft_amt_mean_90d,cc_cnt_sum_90d,cc_cnt_mean_90d,cc_amt_sum_90d,cc_amt_mean_90d,active_prod_mean_90d,active_prod_max_90d,months_active_90d,eft_cnt_sum_180d,eft_cnt_mean_180d,eft_cnt_max_180d,eft_cnt_std_180d,eft_amt_sum_180d,eft_amt_mean_180d,cc_cnt_sum_180d,cc_cnt_mean_180d,cc_amt_sum_180d,cc_amt_mean_180d,active_prod_mean_180d,active_prod_max_180d,months_active_180d,total_months_history,eft_cnt_total,cc_cnt_total,eft_amt_total,cc_amt_total,recency_days,total_transactions_90d,total_amount_90d,eft_cc_ratio_90d,eft_cc_amt_ratio_90d,avg_eft_value_90d,avg_cc_value_90d,recency_risk,is_inactive_90d,activity_score_90d,activity_recent_ratio,gender,age,province,religion,work_type,work_sector,tenure,province_churn_rate_loo,province_count_loo,work_sector_churn_rate_loo,work_sector_count_loo
i64,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,u32,u32,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,str,i64,str,str,str,str,i64,f64,u32,f64,u32
1,2019-02-01 00:00:00,"""test""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1.0,0.5,1.0,0.707107,3.49,1.745,22.0,11.0,136.81,68.405,3.0,3,2,9.0,1.8,5.0,1.923538,21.57,4.314,62.0,12.4,792.2,158.44,3.0,3,5,37,61.0,535.0,662.14,8722.18,31,23.0,140.3,0.043478,0.025325,1.745,5.948261,1.033333,0,68.0,0.319444,"""F""",57,"""ZUI""","""O""","""Full-time""","""Finance""",65,0.140465,28491,0.140171,12656
2,2019-01-01 00:00:00,"""test""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,5.0,2.5,4.0,2.12132,1386.2,693.1,8.0,4.0,29.77,14.885,3.0,3,2,12.0,2.4,5.0,1.949359,2073.3,414.66,17.0,3.4,77.91,15.582,3.0,3,5,36,86.0,66.0,21759.26,333.63,31,13.0,1415.97,0.555556,45.050374,231.033333,3.307778,1.033333,0,34.0,0.433333,"""F""",62,"""NOB""","""M""","""Self-employed""","""Healthcare""",224,0.141195,19767,0.140008,17049
9,2019-03-01 00:00:00,"""test""",2.0,2.0,2.0,0.0,25.36,25.36,1.0,1.0,18.81,18.81,2.0,2,1,7.0,2.333333,4.0,1.527525,43.59,14.53,12.0,4.0,49.64,16.546667,2.0,2,3,9.0,1.8,4.0,1.30384,60.73,12.146,47.0,9.4,409.8,81.96,2.0,2,5,38,129.0,1092.0,20448.12,60672.44,28,19.0,93.23,0.538462,0.860782,5.44875,3.818462,0.933333,0,50.0,0.333333,"""M""",52,"""GRO""","""C""","""Full-time""","""Healthcare""",216,0.141865,4462,0.140008,17049


## 7. Veri Hazırlama

In [ ]:
# Kategorik sütunlar
cat_features = ["gender", "religion", "work_type", "work_sector", "province"]

# Feature listesi (exclude edilecekler)
exclude_cols = ["cust_id", "ref_date", "churn", "split"]
feature_cols = [col for col in train_features.columns if col not in exclude_cols]

# Pandas'a çevir (model için)
X_train = train_features.select(feature_cols).fill_null(0).to_pandas()
y_train = train_features.select("churn").to_pandas()["churn"]
groups_train = train_features.select("cust_id").to_pandas()["cust_id"]

X_test = test_features.select(feature_cols).fill_null(0).to_pandas()
test_ids = test_features.select("cust_id").to_pandas()["cust_id"]

# Kategorikleri string'e çevir (CatBoost için)
for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype(str)
        X_test[col] = X_test[col].astype(str)

print("="*80)
print("VERİ HAZIR")
print("="*80)
print(f"\n📊 X_train: {X_train.shape}")
print(f"📊 y_train: {y_train.shape}")
print(f"📊 X_test: {X_test.shape}")
print(f"\n🎯 Churn distribution:")
print(y_train.value_counts())
print(f"\nChurn rate: {y_train.mean()*100:.2f}%")

VERİ HAZIR

📊 X_train: (133287, 66)
📊 y_train: (133287,)
📊 X_test: (43006, 66)

🎯 Churn distribution:
churn
0    114417
1     18870
Name: count, dtype: int64

Churn rate: 14.16%


## 8. Model Eğitimi - Ensemble Blending

### StratifiedGroupKFold CV

In [12]:
# CV splits
n_splits = 5
cv_splits = list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42).split(
    X_train, y_train, groups=groups_train
))

print(f"✅ {n_splits}-Fold StratifiedGroupKFold CV hazır")

✅ 5-Fold StratifiedGroupKFold CV hazır


### Model Blending

In [24]:
# Model parameters - Overfitting'i daha da azaltmak için
lgbm_params = {
    'n_estimators': 60,  # Daha da düşürdük
    'max_depth': 2,  # Daha da düşürdük
    'learning_rate': 0.005,  # Daha da düşürdük
    'subsample': 0.4,  # Daha da düşürdük
    'colsample_bytree': 0.4,  # Daha da düşürdük
    'min_child_samples': 200,  # Regularization daha da artırıldı
    'reg_alpha': 2.0,  # L1 regularization daha da artırıldı
    'reg_lambda': 10.0,  # L2 regularization daha da artırıldı
    'class_weight': 'balanced',
    'random_state': 42,
    'verbose': -1
}

xgb_params = {
    'n_estimators': 60,  # Daha da düşürdük
    'max_depth': 2,  # Daha da düşürdük
    'learning_rate': 0.005,  # Daha da düşürdük
    'subsample': 0.4,  # Daha da düşürdük
    'colsample_bytree': 0.4,  # Daha da düşürdük
    'min_child_weight': 20,  # Regularization daha da artırıldı
    'gamma': 2.0,  # Regularization daha da artırıldı
    'reg_alpha': 2.0,  # L1 regularization daha da artırıldı
    'reg_lambda': 10.0,  # L2 regularization daha da artırıldı
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
    'random_state': 42,
    'enable_categorical': True,
    'eval_metric': 'auc'
}

# CV training - LightGBM + XGBoost
lgbm_models = []
xgb_models = []

lgbm_oof = np.zeros(len(X_train))
xgb_oof = np.zeros(len(X_train))

lgbm_scores = []
xgb_scores = []

print("="*80)
print("MODEL EĞİTİMİ (ENSEMBLE: LightGBM + XGBoost)")
print("="*80)

for fold, (train_idx, val_idx) in enumerate(cv_splits):
    print(f"\n🔄 Fold {fold+1}/{n_splits}")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # LightGBM
    print("  ⚡ LightGBM eğitiliyor...")
    lgbm = LGBMClassifier(**lgbm_params)
    lgbm.fit(X_tr, y_tr)
    lgbm_pred = lgbm.predict_proba(X_val)[:, 1]
    lgbm_oof[val_idx] = lgbm_pred
    lgbm_auc = roc_auc_score(y_val, lgbm_pred)
    lgbm_scores.append(lgbm_auc)
    lgbm_models.append(lgbm)
    print(f"     ROC-AUC: {lgbm_auc:.4f}")
    
    # XGBoost
    print("  🚀 XGBoost eğitiliyor...")
    xgb = XGBClassifier(**xgb_params)
    xgb.fit(X_tr, y_tr)
    xgb_pred = xgb.predict_proba(X_val)[:, 1]
    xgb_oof[val_idx] = xgb_pred
    xgb_auc = roc_auc_score(y_val, xgb_pred)
    xgb_scores.append(xgb_auc)
    xgb_models.append(xgb)
    print(f"     ROC-AUC: {xgb_auc:.4f}")
    
    # Ensemble prediction (average)
    ensemble_pred = (lgbm_pred + xgb_pred) / 2
    ensemble_auc = roc_auc_score(y_val, ensemble_pred)
    print(f"  🎯 Ensemble ROC-AUC: {ensemble_auc:.4f}")

# Overall OOF scores
ensemble_oof = (lgbm_oof + xgb_oof) / 2

print("\n" + "="*80)
print("CV SONUÇLARI")
print("="*80)
print(f"\n📊 LightGBM CV ROC-AUC: {np.mean(lgbm_scores):.4f} (+/- {np.std(lgbm_scores):.4f})")
print(f"📊 XGBoost CV ROC-AUC:  {np.mean(xgb_scores):.4f} (+/- {np.std(xgb_scores):.4f})")
print(f"\n🎯 Ensemble OOF ROC-AUC: {roc_auc_score(y_train, ensemble_oof):.4f}")
print(f"🎯 Ensemble OOF Accuracy: {accuracy_score(y_train, (ensemble_oof >= 0.5).astype(int)):.4f}")

print("\n✅ Model eğitimi tamamlandı!")

MODEL EĞİTİMİ (ENSEMBLE: LightGBM + XGBoost)

🔄 Fold 1/5
  ⚡ LightGBM eğitiliyor...
     ROC-AUC: 0.7892
  🚀 XGBoost eğitiliyor...
     ROC-AUC: 0.7892
  🚀 XGBoost eğitiliyor...
     ROC-AUC: 0.7753
  🎯 Ensemble ROC-AUC: 0.7839

🔄 Fold 2/5
  ⚡ LightGBM eğitiliyor...
     ROC-AUC: 0.7753
  🎯 Ensemble ROC-AUC: 0.7839

🔄 Fold 2/5
  ⚡ LightGBM eğitiliyor...
     ROC-AUC: 0.7815
  🚀 XGBoost eğitiliyor...
     ROC-AUC: 0.7815
  🚀 XGBoost eğitiliyor...
     ROC-AUC: 0.7730
  🎯 Ensemble ROC-AUC: 0.7783

🔄 Fold 3/5
  ⚡ LightGBM eğitiliyor...
     ROC-AUC: 0.7730
  🎯 Ensemble ROC-AUC: 0.7783

🔄 Fold 3/5
  ⚡ LightGBM eğitiliyor...
     ROC-AUC: 0.7914
  🚀 XGBoost eğitiliyor...
     ROC-AUC: 0.7914
  🚀 XGBoost eğitiliyor...
     ROC-AUC: 0.7834
  🎯 Ensemble ROC-AUC: 0.7878

🔄 Fold 4/5
  ⚡ LightGBM eğitiliyor...
     ROC-AUC: 0.7834
  🎯 Ensemble ROC-AUC: 0.7878

🔄 Fold 4/5
  ⚡ LightGBM eğitiliyor...
     ROC-AUC: 0.7790
  🚀 XGBoost eğitiliyor...
     ROC-AUC: 0.7790
  🚀 XGBoost eğitiliyor...
     R

## 9. Test Set Tahminleri

In [25]:
print("="*80)
print("TEST SET TAHMİNLERİ")
print("="*80)

# Her model ve fold için tahminleri al
lgbm_test_preds = []
xgb_test_preds = []

for fold in range(n_splits):
    print(f"  Fold {fold+1} tahminleri...")
    
    lgbm_pred = lgbm_models[fold].predict_proba(X_test)[:, 1]
    xgb_pred = xgb_models[fold].predict_proba(X_test)[:, 1]
    
    lgbm_test_preds.append(lgbm_pred)
    xgb_test_preds.append(xgb_pred)

# Her modelin fold ortalamalarını al
lgbm_final = np.mean(lgbm_test_preds, axis=0)
xgb_final = np.mean(xgb_test_preds, axis=0)

# Ensemble (2 modelin ortalaması)
final_predictions = (lgbm_final + xgb_final) / 2
final_predictions_binary = (final_predictions >= 0.5).astype(int)

print("\n✅ Tahminler tamamlandı!")
print(f"\n📊 Tahmin İstatistikleri:")
print(f"   Churn=0: {(final_predictions_binary == 0).sum():,}")
print(f"   Churn=1: {(final_predictions_binary == 1).sum():,}")
print(f"   Predicted churn rate: {final_predictions_binary.mean()*100:.2f}%")

TEST SET TAHMİNLERİ
  Fold 1 tahminleri...
  Fold 2 tahminleri...
  Fold 3 tahminleri...
  Fold 4 tahminleri...
  Fold 5 tahminleri...

✅ Tahminler tamamlandı!

📊 Tahmin İstatistikleri:
   Churn=0: 21,548
   Churn=1: 21,458
   Predicted churn rate: 49.90%
  Fold 5 tahminleri...

✅ Tahminler tamamlandı!

📊 Tahmin İstatistikleri:
   Churn=0: 21,548
   Churn=1: 21,458
   Predicted churn rate: 49.90%


## 10. Submission Dosyası

In [26]:
# Submission
submission = pd.DataFrame({
    'cust_id': test_ids,
    'churn': final_predictions_binary
})

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f'submission_vol3_ensemble_{timestamp}.csv'
submission.to_csv(filename, index=False)

print("="*80)
print("✅ SUBMISSION DOSYASI OLUŞTURULDU!")
print("="*80)
print(f"📁 Dosya: {filename}")
print(f"📊 Tahmin sayısı: {len(submission):,}")
print(f"\n📋 İlk 10 tahmin:")
print(submission.head(10))

print("\n🎉 VOL3 TAMAMLANDI!")

✅ SUBMISSION DOSYASI OLUŞTURULDU!
📁 Dosya: submission_vol3_ensemble_20251012_152442.csv
📊 Tahmin sayısı: 43,006

📋 İlk 10 tahmin:
   cust_id  churn
0        1      0
1        2      0
2        9      1
3       15      1
4       19      0
5       21      0
6       26      1
7       32      1
8       33      1
9       36      0

🎉 VOL3 TAMAMLANDI!


## 11. Özet

### ✅ Bu Notebook'ta Yapılanlar:

1. **Polars Kullanımı**: Hızlı veri işleme
2. **Leave-One-Out Encoding**: Data leakage önleme
3. **Time-Based Features**: Zaman pencereli özellikler (30d, 90d, 180d)
4. **Derived Features**: Oranlar, skorlar, risk metrikleri
5. **Ensemble Blending**: LightGBM + XGBoost + CatBoost
6. **StratifiedGroupKFold**: Müşteri bazlı CV

### 🚀 Sonraki Adımlar:

- Historical features (geçmiş oturum davranışları)
- Daha fazla LOO encoding
- Feature selection
- Hyperparameter tuning
- Stacking models